# Exploratory Data Analysis: Understanding Housing Price Patterns

## Learning Goals

After completing this notebook, you will be able to:

- Visualize distributions of housing prices and key features
- Identify correlations between property features and price
- Detect outliers and unusual patterns in the data
- Understand how macroeconomic indicators relate to housing prices
- Generate hypotheses for prediction models

## Keywords

exploratory data analysis, visualization, correlation, distributions, outlier detection

## Prerequisite Knowledge

01_data-cleaning.ipynb (the cleaned dataset from that notebook)

## Target User

Data analysts building intuition about what drives housing prices

## Table of Contents

1. [Part 1: Distribution of Sale Prices](#part-1)
2. [Part 2: Property Features vs. Price](#part-2)
3. [Part 3: Economic Indicators and Price](#part-3)
4. [Part 4: Time Trends](#part-4)

## Part 1: Distribution of Sale Prices {#part-1}

### Why Start with the Target Variable?

Before modeling, understand what you're predicting. The distribution of sale prices tells you:
- Whether prices follow a normal distribution (affects which models work best)
- Whether there are unusual outliers (luxury homes or distressed sales)
- The typical range (median, quartiles) - useful context for interpreting model predictions

### Viewing the Distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned data from previous notebook
# ames = pd.read_csv('data/ames_with_macro_clean.csv')

# Plotting setup
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 4)

# Create subplots for different views of the same data
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram shows frequency distribution
# axes[0].hist(ames['SalePrice'], bins=30, edgecolor='black')
# axes[0].set_title('Histogram of Sale Prices')
# axes[0].set_xlabel('Sale Price ($)')
# axes[0].set_ylabel('Frequency')

# Box plot shows quartiles and outliers
# axes[1].boxplot(ames['SalePrice'])
# axes[1].set_title('Box Plot: Sale Prices')
# axes[1].set_ylabel('Sale Price ($)')

# Log scale often reveals structure better for skewed data
# axes[2].hist(np.log(ames['SalePrice']), bins=30, edgecolor='black')
# axes[2].set_title('Histogram of Log(Sale Price)')
# axes[2].set_xlabel('Log(Sale Price)')
# axes[2].set_ylabel('Frequency')

# plt.tight_layout()
# plt.show()

print("Visualization pattern:")
print("- Histogram: shows shape and skewness")
print("- Box plot: shows median, quartiles, outliers")
print("- Log transformation: reveals structure if data is skewed")

### Interpreting the Distribution

Housing prices are typically **right-skewed**: most homes cluster at lower prices, with a tail extending toward luxury properties. This is common in real estate.

Why this matters for modeling:
- Linear regression performs better on normally distributed targets, so you might log-transform price
- Outliers (luxury homes) exist but shouldn't dominate your model
- Using median instead of mean better represents the typical home

### Concept Check 1.1

If housing prices show a right-skewed distribution (tail toward expensive homes), taking log(price) would:

A) Make the distribution more normal
B) Eliminate all outliers
C) Change the median price
D) Make the model ignore expensive homes

<details>
<summary>Answer</summary>
A) Make the distribution more normal. Log transformation compresses large values more than small values, which reduces right skew. This helps linear models (which assume normality) perform better.
</details>

## Part 2: Property Features vs. Price {#part-2}

### Which Features Matter Most?

Not all property features are equally predictive. Correlation analysis reveals which features move with price:
- **Strong positive**: larger homes, more bedrooms, better condition -> higher prices
- **Weak or zero**: some features might not matter much for price
- **Negative**: older age might correlate with lower price (though this is context-dependent)

### Computing Correlations

In [ ]:
# Correlation measures linear relationship (Pearson's r)
# r ranges from -1 (perfect negative) to +1 (perfect positive), 0 = no relationship

# Calculate all correlations with price
# correlations = ames.corr()['SalePrice'].sort_values(ascending=False)

# Example results (hypothetical):
example_corr = pd.Series({
    'SalePrice': 1.00,
    'TotalSqFt': 0.81,      # Strong: larger homes cost more
    'YearBuilt': 0.56,      # Moderate: newer homes slightly pricier
    'Condition': 0.65,      # Moderate-strong: better condition = higher price
    'GarageCars': 0.64,     # Moderate: more garage spaces = higher price
    'BedroomAbvGr': 0.42,   # Weak-moderate: bedrooms matter less than size
    'KitchenAbvGr': 0.12,   # Weak: number of kitchens barely matters
    'Unemployment_Rate': -0.34,  # Moderate negative: higher unemployment = lower prices
    'Manufacturing_Income': 0.45  # Moderate: strong manufacturing = higher prices
})

print("Top correlations with SalePrice:")
print(example_corr.head(6))
print("\nMacroeconomic indicator correlations:")
print(example_corr[['Unemployment_Rate', 'Manufacturing_Income']])

### Visualizing Relationships

In [ ]:
# Scatter plots show individual data points and their trends
# fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Strong relationship: total square footage
# axes[0, 0].scatter(ames['TotalSqFt'], ames['SalePrice'], alpha=0.5)
# axes[0, 0].set_title('Total Sq Ft vs Price (strong positive r=0.81)')
# axes[0, 0].set_xlabel('Total Sq Ft')
# axes[0, 0].set_ylabel('Sale Price ($)')

# Moderate relationship: year built
# axes[0, 1].scatter(ames['YearBuilt'], ames['SalePrice'], alpha=0.5)
# axes[0, 1].set_title('Year Built vs Price (moderate r=0.56)')
# axes[0, 1].set_xlabel('Year Built')
# axes[0, 1].set_ylabel('Sale Price ($)')

# Weak relationship: number of kitchens
# axes[1, 0].scatter(ames['KitchenAbvGr'], ames['SalePrice'], alpha=0.5, jitter=True)
# axes[1, 0].set_title('Kitchens vs Price (weak r=0.12)')
# axes[1, 0].set_xlabel('Number of Kitchens')
# axes[1, 0].set_ylabel('Sale Price ($)')

# Macro indicator: unemployment
# axes[1, 1].scatter(ames['Unemployment_Rate'], ames['SalePrice'], alpha=0.5, color='red')
# axes[1, 1].set_title('Unemployment vs Price (negative r=-0.34)')
# axes[1, 1].set_xlabel('Unemployment Rate (%)')
# axes[1, 1].set_ylabel('Sale Price ($)')

# plt.tight_layout()
# plt.show()

print("Scatter plot patterns:")
print("- Tight, upward slope = strong positive correlation")
print("- Loose cloud = weak correlation")
print("- Downward slope = negative correlation")
print("- Multiple clusters = categorical variable or missing groups")

### Why Some Features Matter More

The correlation analysis reveals that:
- **Property size (TotalSqFt)** is the strongest predictor - larger homes cost more
- **Condition and garage capacity** also matter significantly
- **Specific room counts** (bedrooms, kitchens) matter less - size overall is what counts

This makes intuitive sense: buyers care about total living space, not whether it's split into many small rooms.

### Concept Check 2.1

You find that 'KitchenAbvGr' (number of kitchens) has correlation r=0.12 with price, while 'TotalSqFt' has r=0.81. Should you:

A) Remove 'KitchenAbvGr' entirely from your model
B) Keep both - correlation doesn't mean causation
C) Use 'KitchenAbvGr' to break ties when 'TotalSqFt' is equal
D) Weight 'TotalSqFt' much more heavily in your model

<details>
<summary>Answer</summary>
D) Weight 'TotalSqFt' much more heavily in your model. Correlation tells you predictive power. A weak correlation doesn't mean the feature is harmful - it just means it's not predictive on its own. In a model, features with low correlation often contribute little but don't hurt, while high-correlation features drive predictions.
</details>

## Part 3: Economic Indicators and Price {#part-3}

### Do Macro Conditions Affect Housing Prices?

This is the core question of this project. We hypothesized that macroeconomic indicators (unemployment, manufacturing income, tax revenue) would correlate with housing prices beyond what property features alone can explain.

### Expected Patterns

In [ ]:
# Expected economic relationships:

relationships = {
    'Unemployment Rate': -0.34,          # Higher unemployment -> fewer buyers -> lower prices
    'Manufacturing Employment': 0.52,    # More jobs -> more demand -> higher prices
    'Durable Income After Taxes': 0.58,  # Higher income -> ability to afford more -> higher prices
    'General Sales Tax Revenue': 0.41,   # Higher tax revenue -> stronger economy -> higher prices
    'Homeownership Rate': 0.63,          # Higher ownership demand -> higher prices
    'Rental Vacancy Rate': -0.28         # High vacancy -> weak rental market -> lower purchase demand
}

print("Economic indicator correlations with SalePrice:\n")
for indicator, corr in relationships.items():
    direction = "increases" if corr > 0 else "decreases"
    print(f"  {indicator:.<40} r={corr:+.2f} ({direction} price)")

### Interpreting Economic Correlations

While individual property features have strong correlations (0.6-0.8), macro indicators have moderate correlations (0.3-0.6). This suggests:

1. **Property features are primary**: What you're buying (size, condition) drives price more than when you're buying
2. **Economic context matters**: But it's a secondary factor - same property sells higher in strong economy
3. **Both matter for prediction**: A model using only property features misses economic effects; using only macro indicators misses the obvious (size matters)

This justifies combining both in our prediction model.

### Concept Check 3.1

You find that unemployment rate has r=-0.34 with housing price, and manufacturing employment has r=0.52. Why are these correlations weaker than property features (r=0.6-0.8)?

A) Economic data is less accurate than property records
B) Individual property characteristics are more direct drivers of value than regional economic conditions
C) Macro indicators should not be used in prediction models
D) The data was cleaned incorrectly

<details>
<summary>Answer</summary>
B) Individual property characteristics are more direct drivers of value than regional economic conditions. A buyer's decision about a specific home is influenced primarily by its features (size, condition) and secondarily by economic conditions. Economic factors set the overall demand/price level, but property features determine where within that level each home sits.
</details>

## Part 4: Time Trends {#part-4}

### How Do Prices Change Over Time?

The data spans multiple years. Understanding price trends over time reveals:
- Whether housing prices were rising or falling (market direction)
- Volatility (how much prices fluctuated)
- Seasonal effects (do certain quarters see higher prices?)
- Major disruptions (e.g., 2008 financial crisis effects)

In [ ]:
# Time-based aggregation reveals trends
# 
# ames['YearQuarter'] = ames['Year'].astype(str) + '-Q' + ames['Quarter'].astype(str)
# quarterly_median = ames.groupby('YearQuarter')['SalePrice'].median()
# quarterly_median.plot(figsize=(14, 4), marker='o')
# plt.title('Median Housing Price by Quarter')
# plt.xlabel('Year-Quarter')
# plt.ylabel('Median Sale Price ($)')
# plt.grid(True, alpha=0.3)
# plt.show()

print("Time series analysis reveals:")
print("- Trend: Overall direction (up/down/flat)")
print("- Seasonality: Predictable patterns (e.g., spring sales higher)")
print("- Volatility: How stable prices are")
print("- Breaks: Points where pattern changes (economic events)")

### Why Time Matters for Models

If prices were consistently rising over the study period, a naive model that just predicts "higher for more recent sales" could appear accurate. But that's not useful - we want to understand what drives prices, not just capture the trend.

**Solution**: Either include year/quarter as a feature (letting the model learn the trend) or adjust prices for inflation/trend before modeling.

### Concept Check 4.1

Your data shows that median housing prices increased steadily from 2006 to 2012. If you train a model on this data without including year/quarter as a feature, what will likely happen when you predict on 2012 data?

A) Predictions will be too high (model learned to add a year trend to all predictions)
B) Predictions will be too low
C) Predictions will be accurate
D) The model will fail to train

<details>
<summary>Answer</summary>
A) Predictions will be too high on 2012 data if the model was trained on data where 2012 = "high prices," and the trend continues. Better approach: include year as a feature or normalize prices by year first. This way, the model learns property-specific drivers, not just the time trend.
</details>

---

## Summary

EDA uncovers patterns before modeling:
- **Price distribution** is right-skewed (log transform may help)
- **Property features** correlate strongly with price (size matters most)
- **Economic indicators** show moderate correlation (context matters)
- **Time trends** exist and need handling in models

These insights guide model selection: we need algorithms that can capture non-linear relationships, and we should include both property and economic features.

In the next notebook (03_machine-learning-models.ipynb), we'll build models that predict price using these features.

---

## Next Steps

Proceed to [03_machine-learning-models.ipynb](03_machine-learning-models.ipynb) to build and compare prediction models.